# COMET Cell-Based Transcript Assignment for STOmics

This notebook assigns STOmics transcripts to COMET-defined cells using two approaches:

**Path A (SAW-native):** Rasterize COMET cell polygons → TIFF mask → geftools/stereopy → cellbin GEF  
**Path B (Python direct):** Rasterize COMET cell polygons → mask lookup on bin1 GEF → AnnData  

Both paths merge COMET protein intensities and phenotype annotations into the output.

**Pilot sample:** SO34 (MLA, chip A03979E2)  
**Validation sample:** SO4 (Ovarian, chip C03027C4)

---

## 1. Setup & Configuration

In [ ]:
import sys
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tiff

# Add repo root to path
REPO_DIR = Path(r'T:/0_Organizational/Git/MDACC-STOmics-COMET-MSI')
sys.path.insert(0, str(REPO_DIR))

from comet.alignment_utils import (
    DefaultPaths, ALL_SAMPLES, OVARIAN_WITH_STOMICS,
    PROTEIN_GENE_MAP, PHENOTYPE_MARKERS,
    check_sample_data, load_geojson, geojson_centroids,
    load_stomics_cellbin_gef,
    rasterize_geojson_to_mask,
    generate_cellbin_with_comet_mask,
    aggregate_transcripts_by_mask,
    annotate_with_comet_phenotypes,
    validate_protein_gene_correlation,
    compute_alignment_metrics,
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)

plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print('Imports OK')

In [ ]:
# ============= CONFIGURATION =============

SAMPLE_ID = 'SO34'
CHIP_ID = 'A03979E2'
DISEASE = 'MLA'

# Output directory
OUTPUT_DIR = Path('T:/Sammy Data/projects/out/comet_stomics_alignment')
SAMPLE_DIR = OUTPUT_DIR / f'{SAMPLE_ID}_{CHIP_ID}'
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# Input paths
WARPED_GEOJSON_PATH = SAMPLE_DIR / f'{SAMPLE_ID}_warped_segmentations.geojson'
STOMICS_DAPI_PATH = DefaultPaths.stomics_dapi_path(CHIP_ID)
TISSUE_GEF_PATH = DefaultPaths.stomics_tissue_gef_path(CHIP_ID)
COMET_PROTEIN_PATH = Path(f'T:/Sammy Data/projects/out/out_comet/{SAMPLE_ID}_BS_protein_combined.parquet')
STOMICS_CELLBIN_PATH = DefaultPaths.stomics_cellbin_path(CHIP_ID)

print(f'Sample: {SAMPLE_ID} ({DISEASE})')
print(f'Chip: {CHIP_ID}')
print(f'Output: {SAMPLE_DIR}')
print()
print('Input files:')
for name, path in [('Warped GeoJSON', WARPED_GEOJSON_PATH),
                    ('STOmics DAPI', STOMICS_DAPI_PATH),
                    ('Tissue GEF', TISSUE_GEF_PATH),
                    ('COMET Protein', COMET_PROTEIN_PATH),
                    ('STOmics Cellbin', STOMICS_CELLBIN_PATH)]:
    exists = path.exists()
    size = f'({path.stat().st_size / 1e6:.1f} MB)' if exists else ''
    print(f'  {name}: {"OK" if exists else "MISSING"} {size}')
    print(f'    {path}')

## 2. Load Warped GeoJSON & Verify Alignment

The warped GeoJSON was produced by `script03` using VALIS registration.
We verify it's in STOmics coordinate space.

In [ ]:
# Load warped COMET segmentations
warped_geojson = load_geojson(WARPED_GEOJSON_PATH)
n_cells = len(warped_geojson['features'])
centroids, labels = geojson_centroids(warped_geojson)

print(f'Warped cells: {n_cells:,}')
print(f'Centroid X range: [{centroids[:, 0].min():.0f}, {centroids[:, 0].max():.0f}]')
print(f'Centroid Y range: [{centroids[:, 1].min():.0f}, {centroids[:, 1].max():.0f}]')

# Load STOmics DAPI for reference dimensions
dapi = tiff.imread(str(STOMICS_DAPI_PATH))
DAPI_SHAPE = dapi.shape
print(f'\nSTOmics DAPI shape: {DAPI_SHAPE}')
print(f'Coordinates fit within DAPI: '
      f'X={centroids[:, 0].max() < DAPI_SHAPE[1]}, '
      f'Y={centroids[:, 1].max() < DAPI_SHAPE[0]}')

# Quick spatial plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(dapi, cmap='gray')
axes[0].set_title(f'STOmics DAPI ({CHIP_ID})')

axes[1].scatter(centroids[:, 0], centroids[:, 1], s=0.1, alpha=0.2, c='red')
axes[1].set_xlim(0, DAPI_SHAPE[1])
axes[1].set_ylim(DAPI_SHAPE[0], 0)
axes[1].set_aspect('equal')
axes[1].set_title(f'Warped COMET Centroids ({n_cells:,} cells)')

plt.suptitle(f'{SAMPLE_ID}: Alignment Verification', fontweight='bold')
plt.tight_layout()
plt.show()
del dapi

## 3. Rasterize to TIFF Cell Mask

Convert warped COMET polygons into a labeled uint32 TIFF mask.
Each pixel value = COMET cell label. Background = 0.

In [ ]:
mask_path = SAMPLE_DIR / f'{SAMPLE_ID}_{CHIP_ID}_comet_mask.tif'

if mask_path.exists():
    print(f'Mask already exists: {mask_path}')
    print(f'Size: {mask_path.stat().st_size / 1e6:.1f} MB')
    mask = tiff.imread(str(mask_path))
else:
    mask = rasterize_geojson_to_mask(
        warped_geojson,
        mask_shape=DAPI_SHAPE,
        output_path=mask_path
    )

# Validate mask
unique_labels = np.unique(mask)
n_mask_cells = len(unique_labels) - 1  # exclude 0
print(f'\nMask shape: {mask.shape}, dtype: {mask.dtype}')
print(f'Unique cell labels: {n_mask_cells:,}')
print(f'Max label: {mask.max()}')
print(f'GeoJSON cells: {n_cells:,}')
print(f'Label coverage: {n_mask_cells/n_cells*100:.1f}%')
print(f'Nonzero pixels: {(mask > 0).sum():,} / {mask.size:,} '
      f'({(mask > 0).mean()*100:.1f}%)')

In [ ]:
# Visual inspection of the mask
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Full mask (binary view)
axes[0].imshow(mask > 0, cmap='gray')
axes[0].set_title(f'COMET Cell Mask (binary, {n_mask_cells:,} cells)')

# Zoomed region with colored labels
cx, cy = int(np.median(centroids[:, 0])), int(np.median(centroids[:, 1]))
r = 1000
zoom = mask[max(0,cy-r):cy+r, max(0,cx-r):cx+r]
axes[1].imshow(zoom, cmap='nipy_spectral', interpolation='nearest')
axes[1].set_title(f'Zoomed Mask (center {r*2}x{r*2}px)')

plt.suptitle(f'{SAMPLE_ID}: Rasterized COMET Cell Mask', fontweight='bold')
plt.tight_layout()
plt.savefig(str(SAMPLE_DIR / f'{SAMPLE_ID}_mask_inspection.png'), dpi=150)
plt.show()

## 4. Path A: SAW-Native Cellbin GEF Generation

Use stereopy `cell_correct` or geftools CLI to generate a SAW-compatible
cellbin GEF from the COMET mask + bin1 tissue GEF.

**Note:** geftools re-numbers cell IDs. Original COMET labels are lost.
We re-map via centroid spatial join.

In [ ]:
# Generate cellbin GEF via SAW-native tools
cellbin_output_dir = SAMPLE_DIR / 'comet_cellbin'

cellbin_result = generate_cellbin_with_comet_mask(
    tissue_gef_path=TISSUE_GEF_PATH,
    mask_path=mask_path,
    output_dir=cellbin_output_dir,
    method='FAST',
    expand_distance=0,
)

print(f'\nResult: {cellbin_result["status"]}')
if cellbin_result.get('backend'):
    print(f'Backend: {cellbin_result["backend"]}')
for k, v in cellbin_result.items():
    if 'gef' in k.lower():
        print(f'{k}: {v}')

In [ ]:
# Load the generated cellbin GEF into AnnData
path_a_adata = None

cellbin_gef_path = cellbin_result.get('adjusted_cellbin_gef') or \
                   cellbin_result.get('raw_cellbin_gef') or \
                   cellbin_result.get('cellbin_gef')

if cellbin_gef_path and Path(cellbin_gef_path).exists():
    path_a_adata = load_stomics_cellbin_gef(cellbin_gef_path)
    print(f'Path A AnnData: {path_a_adata.shape}')
    print(f'  Cells: {path_a_adata.n_obs:,}')
    print(f'  Genes: {path_a_adata.n_vars:,}')
    print(f'  Total transcripts: {path_a_adata.X.sum():.0f}')
else:
    print('Path A: cellbin GEF not generated (stereopy/geftools not available).')
    print('Continuing with Path B (Python direct) only.')

## 5. Path B: Python Direct Transcript Assignment

Load bin1 GEF transcripts via h5py, look up cell assignment from the
rasterized mask, and aggregate counts per COMET cell.

**Advantage:** Preserves original COMET cell labels. No external dependencies.

In [ ]:
# Direct transcript-to-cell assignment
path_b_adata = aggregate_transcripts_by_mask(
    tissue_gef_path=TISSUE_GEF_PATH,
    mask=mask,
    geojson_data=warped_geojson,
)

print(f'\nPath B AnnData: {path_b_adata.shape}')
print(f'  Cells: {path_b_adata.n_obs:,}')
print(f'  Genes: {path_b_adata.n_vars:,}')
print(f'  Total transcripts: {path_b_adata.X.sum():.0f}')
print(f'  DNBs assigned: {path_b_adata.uns["total_dnbs_assigned"]:,}')
print(f'  DNBs outside cells: {path_b_adata.uns["total_dnbs_outside"]:,}')
print(f'\nPer-cell stats:')
print(f'  Median transcripts/cell: {path_b_adata.obs["n_transcripts"].median():.0f}')
print(f'  Median genes/cell: {path_b_adata.obs["n_genes"].median():.0f}')
print(f'  Mean transcripts/cell: {path_b_adata.obs["n_transcripts"].mean():.0f}')

In [ ]:
# Save Path B output
path_b_h5ad = SAMPLE_DIR / f'{SAMPLE_ID}_{CHIP_ID}_comet_transcripts.h5ad'
path_b_adata.write_h5ad(str(path_b_h5ad))
print(f'Saved: {path_b_h5ad}')

# Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(path_b_adata.obs['n_transcripts'], bins=100, alpha=0.7,
             color='steelblue', edgecolor='white')
axes[0].set_xlabel('Transcripts per cell')
axes[0].set_ylabel('Count')
axes[0].set_title('Transcript Distribution')
axes[0].axvline(path_b_adata.obs['n_transcripts'].median(), c='red', ls='--',
                label=f'median={path_b_adata.obs["n_transcripts"].median():.0f}')
axes[0].legend()

axes[1].hist(path_b_adata.obs['n_genes'], bins=100, alpha=0.7,
             color='coral', edgecolor='white')
axes[1].set_xlabel('Genes per cell')
axes[1].set_ylabel('Count')
axes[1].set_title('Gene Detection Distribution')
axes[1].axvline(path_b_adata.obs['n_genes'].median(), c='red', ls='--',
                label=f'median={path_b_adata.obs["n_genes"].median():.0f}')
axes[1].legend()

if 'area_px' in path_b_adata.obs.columns:
    areas = path_b_adata.obs['area_px']
    axes[2].scatter(areas, path_b_adata.obs['n_transcripts'], s=0.5, alpha=0.1)
    axes[2].set_xlabel('Cell Area (px)')
    axes[2].set_ylabel('Transcripts')
    axes[2].set_title('Area vs Transcripts')
    axes[2].set_xlim(0, np.percentile(areas.dropna(), 99))

plt.suptitle(f'{SAMPLE_ID}: Path B — Transcript Assignment Quality', fontweight='bold')
plt.tight_layout()
plt.savefig(str(SAMPLE_DIR / f'{SAMPLE_ID}_path_b_distributions.png'), dpi=150)
plt.show()

## 6. Merge COMET Phenotype Annotations

Add COMET protein intensities (24 channels) and hierarchical phenotype
classifications to the AnnData objects.

In [ ]:
# Annotate Path B (label-based matching — exact)
if COMET_PROTEIN_PATH.exists():
    path_b_adata = annotate_with_comet_phenotypes(
        path_b_adata,
        comet_protein_path=COMET_PROTEIN_PATH,
        match_method='label',
    )
    print(f'\nPath B phenotype distribution:')
    print(path_b_adata.obs['phenotype'].value_counts())
    print(f'\nProtein columns added: {sum(1 for c in path_b_adata.obs.columns if c.startswith("protein_"))}')
else:
    print(f'COMET protein data not found: {COMET_PROTEIN_PATH}')

In [ ]:
# Annotate Path A (spatial matching — KD-tree) if available
if path_a_adata is not None and COMET_PROTEIN_PATH.exists():
    path_a_adata = annotate_with_comet_phenotypes(
        path_a_adata,
        comet_protein_path=COMET_PROTEIN_PATH,
        warped_geojson=warped_geojson,
        match_method='spatial',
        match_threshold=30,
    )
    print(f'\nPath A phenotype distribution:')
    print(path_a_adata.obs['phenotype'].value_counts())
else:
    print('Path A not available or protein data missing — skipping annotation')

In [ ]:
# Save annotated AnnData
path_b_annotated_h5ad = SAMPLE_DIR / f'{SAMPLE_ID}_{CHIP_ID}_comet_transcripts_annotated.h5ad'
path_b_adata.write_h5ad(str(path_b_annotated_h5ad))
print(f'Saved annotated Path B: {path_b_annotated_h5ad}')

if path_a_adata is not None:
    path_a_annotated_h5ad = SAMPLE_DIR / f'{SAMPLE_ID}_{CHIP_ID}_comet_cellbin_annotated.h5ad'
    path_a_adata.write_h5ad(str(path_a_annotated_h5ad))
    print(f'Saved annotated Path A: {path_a_annotated_h5ad}')

## 7. Comparison & Validation

Compare Path A vs Path B, and both against the existing STOmics cellbin.

In [ ]:
# Load existing STOmics cellbin for comparison
stomics_cellbin_adata = None
if STOMICS_CELLBIN_PATH.exists():
    stomics_cellbin_adata = load_stomics_cellbin_gef(STOMICS_CELLBIN_PATH)
    print(f'STOmics cellbin: {stomics_cellbin_adata.shape}')
else:
    print('STOmics cellbin not available for comparison')

# Comparison table
rows = []
for name, adata in [('Path B (Python)', path_b_adata),
                     ('Path A (SAW)', path_a_adata),
                     ('STOmics (orig)', stomics_cellbin_adata)]:
    if adata is None:
        continue
    total_tx = adata.X.sum()
    rows.append({
        'Method': name,
        'Cells': adata.n_obs,
        'Genes': adata.n_vars,
        'Total Transcripts': int(total_tx),
        'Median Tx/Cell': np.median(np.array(adata.X.sum(axis=1)).ravel()),
        'Median Genes/Cell': np.median(np.array((adata.X > 0).sum(axis=1)).ravel()),
    })

comparison_df = pd.DataFrame(rows)
print('\n' + '=' * 80)
print('COMPARISON: COMET vs STOmics Cell Segmentation')
print('=' * 80)
print(comparison_df.to_string(index=False))
print('=' * 80)

In [ ]:
# Protein-gene correlation validation (Path B)
if COMET_PROTEIN_PATH.exists() and 'protein_CD4' in path_b_adata.obs.columns:
    comet_protein_df = pd.read_parquet(str(COMET_PROTEIN_PATH))
    
    # Align by cell label
    n_min = min(len(comet_protein_df), path_b_adata.n_obs)
    corr_results = validate_protein_gene_correlation(
        path_b_adata, comet_protein_df.iloc[:n_min]
    )

    if len(corr_results) > 0:
        print(f'\nTop protein-gene correlations:')
        print(corr_results.head(10).to_string(index=False))
        
        # Plot top 6
        n_plot = min(6, len(corr_results))
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.ravel()
        
        for i, (_, row) in enumerate(corr_results.head(n_plot).iterrows()):
            ax = axes[i]
            prot_col = f'protein_{row["protein"]}'
            gene_idx = list(path_b_adata.var_names).index(row['gene'])
            prot_vals = path_b_adata.obs[prot_col].values
            gene_vals = path_b_adata.X[:, gene_idx]
            if hasattr(gene_vals, 'toarray'):
                gene_vals = gene_vals.toarray().ravel()
            gene_vals = np.asarray(gene_vals).ravel()
            
            valid = ~np.isnan(prot_vals)
            ax.scatter(prot_vals[valid], gene_vals[valid], s=0.5, alpha=0.1)
            ax.set_xlabel(f'{row["protein"]} (protein)')
            ax.set_ylabel(f'{row["gene"]} (gene)')
            ax.set_title(f'rho={row["spearman_rho"]:.3f}')
        
        for i in range(n_plot, len(axes)):
            axes[i].set_visible(False)
        
        plt.suptitle(f'{SAMPLE_ID}: Protein-Gene Correlations (Path B)', fontweight='bold')
        plt.tight_layout()
        plt.savefig(str(SAMPLE_DIR / f'{SAMPLE_ID}_protein_gene_correlations_pathB.png'), dpi=150)
        plt.show()

## 8. Exploratory Analysis

UMAP and spatial plots colored by COMET phenotype.

In [ ]:
import scanpy as sc

# Use Path B data for exploratory analysis
adata_explore = path_b_adata.copy()

# Basic preprocessing
sc.pp.filter_cells(adata_explore, min_genes=5)
sc.pp.filter_genes(adata_explore, min_cells=10)
sc.pp.normalize_total(adata_explore, target_sum=1e4)
sc.pp.log1p(adata_explore)
sc.pp.highly_variable_genes(adata_explore, n_top_genes=2000)

print(f'After filtering: {adata_explore.shape}')
print(f'Phenotype distribution after filtering:')
if 'phenotype' in adata_explore.obs.columns:
    print(adata_explore.obs['phenotype'].value_counts())

In [ ]:
# PCA + UMAP
sc.tl.pca(adata_explore, n_comps=30)
sc.pp.neighbors(adata_explore, n_pcs=20)
sc.tl.umap(adata_explore)

# UMAP colored by phenotype
if 'phenotype' in adata_explore.obs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    sc.pl.umap(adata_explore, color='phenotype', ax=axes[0], show=False,
               title=f'{SAMPLE_ID}: UMAP by COMET Phenotype')
    
    # Spatial plot colored by phenotype
    if 'spatial' in adata_explore.obsm:
        coords = adata_explore.obsm['spatial']
        phenotypes = adata_explore.obs['phenotype']
        unique_pheno = phenotypes.unique()
        colors = plt.cm.tab20(np.linspace(0, 1, len(unique_pheno)))
        pheno_colors = {p: colors[i] for i, p in enumerate(unique_pheno)}
        
        for pheno in unique_pheno:
            mask_p = phenotypes == pheno
            axes[1].scatter(coords[mask_p, 0], coords[mask_p, 1],
                           s=0.3, alpha=0.3, c=[pheno_colors[pheno]],
                           label=pheno)
        axes[1].set_aspect('equal')
        axes[1].legend(markerscale=5, fontsize=8, loc='upper right')
        axes[1].set_title(f'{SAMPLE_ID}: Spatial Phenotype Map')
        axes[1].set_xlabel('X (STOmics pixels)')
        axes[1].set_ylabel('Y (STOmics pixels)')
    
    plt.tight_layout()
    plt.savefig(str(SAMPLE_DIR / f'{SAMPLE_ID}_phenotype_umap_spatial.png'), dpi=150)
    plt.show()

In [ ]:
# Differential expression: CD45+ immune vs CD45- non-immune
if 'phenotype' in adata_explore.obs.columns:
    immune_types = {'Treg', 'CD4+ T cell', 'CD8+ T cell', 'Macrophage',
                    'M2 Macrophage', 'B cell', 'NK cell', 'DC', 'Neutrophil',
                    'Immune (other)'}
    adata_explore.obs['immune_status'] = adata_explore.obs['phenotype'].apply(
        lambda x: 'Immune' if x in immune_types else 'Non-immune'
    )
    
    n_immune = (adata_explore.obs['immune_status'] == 'Immune').sum()
    n_non = (adata_explore.obs['immune_status'] == 'Non-immune').sum()
    print(f'Immune cells: {n_immune:,}')
    print(f'Non-immune cells: {n_non:,}')
    
    if n_immune >= 50 and n_non >= 50:
        sc.tl.rank_genes_groups(adata_explore, 'immune_status', method='wilcoxon')
        sc.pl.rank_genes_groups(adata_explore, n_genes=15,
                                title=f'{SAMPLE_ID}: Immune vs Non-immune DE Genes')
        plt.savefig(str(SAMPLE_DIR / f'{SAMPLE_ID}_immune_de_genes.png'), dpi=150)
        plt.show()
    else:
        print('Insufficient cells for differential expression analysis')

## 9. Validation on SO4 (Ovarian)

Run the same pipeline on an Ovarian sample to confirm cross-tissue robustness.

In [ ]:
VAL_SAMPLE = 'SO4'
VAL_CHIP = 'C03027C4'

val_warped_path = OUTPUT_DIR / f'{VAL_SAMPLE}_{VAL_CHIP}' / f'{VAL_SAMPLE}_warped_segmentations.geojson'
val_tissue_gef = DefaultPaths.stomics_tissue_gef_path(VAL_CHIP)
val_dapi = DefaultPaths.stomics_dapi_path(VAL_CHIP)
val_protein = Path(f'T:/Sammy Data/projects/out/out_comet/{VAL_SAMPLE}_BS_protein_combined.parquet')

print(f'Validation sample: {VAL_SAMPLE} ({VAL_CHIP})')
for name, path in [('Warped GeoJSON', val_warped_path),
                    ('Tissue GEF', val_tissue_gef),
                    ('DAPI', val_dapi),
                    ('Protein', val_protein)]:
    print(f'  {name}: {"OK" if path.exists() else "MISSING"} — {path}')

can_validate = val_warped_path.exists() and val_tissue_gef.exists() and val_dapi.exists()

In [ ]:
if can_validate:
    val_dir = OUTPUT_DIR / f'{VAL_SAMPLE}_{VAL_CHIP}'
    val_dir.mkdir(parents=True, exist_ok=True)
    
    # Load warped GeoJSON
    val_geojson = load_geojson(val_warped_path)
    
    # Get DAPI shape
    val_dapi_img = tiff.imread(str(val_dapi))
    val_dapi_shape = val_dapi_img.shape
    del val_dapi_img
    print(f'SO4 DAPI shape: {val_dapi_shape}')
    
    # Rasterize
    val_mask_path = val_dir / f'{VAL_SAMPLE}_{VAL_CHIP}_comet_mask.tif'
    if val_mask_path.exists():
        val_mask = tiff.imread(str(val_mask_path))
    else:
        val_mask = rasterize_geojson_to_mask(val_geojson, val_dapi_shape, val_mask_path)
    
    # Path B: direct transcript assignment
    val_adata = aggregate_transcripts_by_mask(val_tissue_gef, val_mask, val_geojson)
    print(f'\nSO4 Path B: {val_adata.shape}')
    print(f'  Median tx/cell: {val_adata.obs["n_transcripts"].median():.0f}')
    print(f'  Median genes/cell: {val_adata.obs["n_genes"].median():.0f}')
    
    # Annotate
    if val_protein.exists():
        val_adata = annotate_with_comet_phenotypes(
            val_adata, val_protein, match_method='label'
        )
    
    # Save
    val_h5ad = val_dir / f'{VAL_SAMPLE}_{VAL_CHIP}_comet_transcripts_annotated.h5ad'
    val_adata.write_h5ad(str(val_h5ad))
    print(f'Saved: {val_h5ad}')
else:
    print('Cannot run validation — missing input files.')
    print('Run script03 alignment pipeline on SO4 first.')

In [ ]:
# Cross-sample comparison
if can_validate:
    rows = [
        {'Sample': f'{SAMPLE_ID} (MLA)',
         'Cells': path_b_adata.n_obs,
         'Genes': path_b_adata.n_vars,
         'Median Tx/Cell': path_b_adata.obs['n_transcripts'].median(),
         'Median Genes/Cell': path_b_adata.obs['n_genes'].median()},
        {'Sample': f'{VAL_SAMPLE} (Ovarian)',
         'Cells': val_adata.n_obs,
         'Genes': val_adata.n_vars,
         'Median Tx/Cell': val_adata.obs['n_transcripts'].median(),
         'Median Genes/Cell': val_adata.obs['n_genes'].median()},
    ]
    cross_df = pd.DataFrame(rows)
    print('\nCross-Sample Comparison:')
    print('=' * 70)
    print(cross_df.to_string(index=False))
    print('=' * 70)

## 10. Summary

Output files and pipeline recap.

In [ ]:
print('=' * 70)
print(f'CELLBIN GENERATION COMPLETE: {SAMPLE_ID} ({DISEASE})')
print('=' * 70)
print(f'\nPath B (Python Direct):')
print(f'  AnnData: {path_b_adata.shape}')
print(f'  Total transcripts: {path_b_adata.X.sum():.0f}')
if 'phenotype' in path_b_adata.obs.columns:
    print(f'  Phenotypes assigned: {(path_b_adata.obs["phenotype"] != "Unclassified").sum():,}')

if path_a_adata is not None:
    print(f'\nPath A (SAW Native):')
    print(f'  AnnData: {path_a_adata.shape}')
    print(f'  Total transcripts: {path_a_adata.X.sum():.0f}')

print(f'\nOutput files in {SAMPLE_DIR}:')
for f in sorted(SAMPLE_DIR.rglob('*')):
    if f.is_file() and not f.name.startswith('.'):
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.relative_to(SAMPLE_DIR)} ({size_mb:.1f} MB)')

print('\n' + '=' * 70)
print('DONE')
print('=' * 70)